# cap10_wf2020_failed_fold_objective_audit_v1

Objective: inspect the remaining `wf_2020` failure after the three-fold cap10 BC/checkpoint metric replay, and decide whether the next branch is allowed to run full retained training as-is.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison: `cap10_three_fold_checkpoint_metric_replay_v2`.
- Reference baseline: `soxlbacktest_qlib_buysell_reference` / `legacy_default`.
- Role: `diagnose`.
- Target window: `wf_2020` plus cross-fold guard context from `wf_2022` and `wf_2023_2024`.
- Changed layer: `model_selection`, `training_objective`, `target_design`.
- External reference: legacy SOXLBacktest buy/sell path and qlib-aligned SOXL replay sidecars.

Hypothesis: the cap10 joint target can learn risk-control labels, but the `wf_2020` fold fails because the current target contract under-represents post-crash bull re-entry participation. Directly lengthening PPO or retaining an actor before a target-design correction risks repeating the existing drawdown/participation tradeoff.

Acceptance for this diagnostic: identify the failed metric source, quantify SOXLBacktest 2020 stage behavior, quantify target-contract participation coverage, and write a next-training contract with stop rules.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists():
    raise RuntimeError(f'could not locate repo root from {Path.cwd()}')
EXPERIMENT = 'cap10_wf2020_failed_fold_objective_audit_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

METRIC_REPLAY = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_three_fold_checkpoint_metric_replay_v2'
BC_SCREEN = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_three_fold_bc_slice_screen_v1'
WF2020_CONTRACT = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_wf2020_soxlbacktest_target_contract_v1'
SOXLBT = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_qlib_walkforward_baseline_v1'
HANDOFF = ROOT / 'runtime' / 'model_selection' / 'card' / 'retraining_gate_gap_handoff_v1'

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
CURRENT_BEST_YTD_RETURN_PCT = 224.361539
SOXLBT_REFERENCE_YTD_RETURN_PCT = 132.077978

In [2]:
required_paths = {
    'metric_candidates': METRIC_REPLAY / 'checkpoint_candidate_metrics.csv',
    'metric_selection': METRIC_REPLAY / 'checkpoint_selection_by_fold.csv',
    'bc_state_slice': BC_SCREEN / 'state_slice_bc_metrics.csv',
    'bc_fold_metrics': BC_SCREEN / 'fold_bc_metrics.csv',
    'combined_contract': BC_SCREEN / 'combined_target_contract.csv',
    'wf2020_contract': WF2020_CONTRACT / 'soxlbacktest_wf2020_target_contract.csv',
    'soxlbt_daily': SOXLBT / 'candidate_daily.csv',
    'soxlbt_trades': SOXLBT / 'candidate_trades.csv',
    'handoff': HANDOFF / 'training_objective_handoff.json',
}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'missing required sidecars: {missing}')

candidate_metrics = pd.read_csv(required_paths['metric_candidates'])
selection_by_fold = pd.read_csv(required_paths['metric_selection'])
bc_state_slice = pd.read_csv(required_paths['bc_state_slice'])
bc_fold_metrics = pd.read_csv(required_paths['bc_fold_metrics'])
combined_contract = pd.read_csv(required_paths['combined_contract'], parse_dates=['datetime'])
wf2020_contract = pd.read_csv(required_paths['wf2020_contract'], parse_dates=['datetime'])
soxlbt_daily = pd.read_csv(required_paths['soxlbt_daily'], parse_dates=['date'])
soxlbt_trades = pd.read_csv(required_paths['soxlbt_trades'], parse_dates=['date'])
handoff = json.loads(required_paths['handoff'].read_text(encoding='utf-8'))

print('metric rows', len(candidate_metrics), 'selection folds', selection_by_fold['fold'].tolist())
print('contract rows', len(combined_contract), 'wf2020 rows', len(wf2020_contract))

metric rows 12 selection folds ['wf_2020', 'wf_2022', 'wf_2023_2024']
contract rows 3731 wf2020 rows 2470


In [3]:
metric_gap = selection_by_fold.copy()
metric_gap['relative_gap_to_target_pp'] = (TARGET_RELATIVE_RETURN_PP - metric_gap['selected_relative_return']).clip(lower=0)
metric_gap['maxdd_gap_to_target_pp'] = (metric_gap['selected_max_dd'] - TARGET_MAX_DD_PCT).clip(lower=0)
metric_gap['primary_failure'] = metric_gap.apply(
    lambda row: 'relative_underparticipation' if row['relative_gap_to_target_pp'] > 0 and row['maxdd_gap_to_target_pp'] <= 0 else (
        'drawdown_control' if row['maxdd_gap_to_target_pp'] > 0 else 'none'
    ),
    axis=1,
)
metric_gap.to_csv(CARD / 'wf_fold_metric_gap.csv', index=False)
metric_gap

,fold,status,selected_candidate,selected_kind,selected_iter,selected_target_gate,selected_drift_gate,selected_candidate_gate,selected_loss,selected_relative_return,selected_max_dd,bc_relative_return,bc_max_dd,candidate_count,relative_gap_to_target_pp,maxdd_gap_to_target_pp,primary_failure
0,wf_2020,evaluated,bc_initialization,bc_initialization,0,False,True,False,10.758434,-292.753021,14.984701,-292.753021,14.984701,4,322.753021,0.0,relative_underparticipation
1,wf_2022,evaluated,bc_initialization,bc_initialization,0,True,True,True,0.000000,65.864820,18.802284,65.864820,18.802284,4,0.000000,0.0,none
2,wf_2023_2024,evaluated,ppo_retained_iter_4,ppo_retained,4,True,True,True,0.004000,35.566229,8.955769,25.313413,19.933946,4,0.000000,0.0,none


In [4]:
wf2020_candidates = candidate_metrics[candidate_metrics['fold'].eq('wf_2020')].copy()
wf2020_candidates['return_gap_to_soxlbt_ytd_reference_pp'] = SOXLBT_REFERENCE_YTD_RETURN_PCT - wf2020_candidates['test_return']
wf2020_candidates['return_gap_to_current_best_floor_pp'] = CURRENT_BEST_YTD_RETURN_PCT * 0.95 - wf2020_candidates['test_return']
wf2020_candidates['relative_gap_to_target_pp'] = (TARGET_RELATIVE_RETURN_PP - wf2020_candidates['test_relative_return']).clip(lower=0)
wf2020_candidates['maxdd_gap_to_target_pp'] = (wf2020_candidates['test_max_dd'] - TARGET_MAX_DD_PCT).clip(lower=0)
wf2020_candidates.to_csv(CARD / 'wf2020_checkpoint_candidate_gap.csv', index=False)
wf2020_candidates[['candidate','candidate_kind','iter','test_return','test_max_dd','test_avg_pos','test_buy_and_hold_return','test_relative_return','relative_delta_vs_bc','max_dd_delta_vs_bc','target_gate','drift_gate','candidate_gate','relative_gap_to_target_pp','maxdd_gap_to_target_pp']]

,candidate,candidate_kind,iter,test_return,test_max_dd,test_avg_pos,test_buy_and_hold_return,test_relative_return,relative_delta_vs_bc,max_dd_delta_vs_bc,target_gate,drift_gate,candidate_gate,relative_gap_to_target_pp,maxdd_gap_to_target_pp
0,bc_initialization,bc_initialization,0,40.580326,14.984701,0.405426,333.333347,-292.753021,0.000000,0.000000,False,True,False,322.753021,0.000000
1,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,40.580326,14.984701,0.405426,333.333347,-292.753021,0.000000,0.000000,False,True,False,322.753021,0.000000
2,ppo_retained_iter_4,ppo_retained,4,61.324050,20.918285,0.358682,333.333347,-272.009298,20.743724,5.933585,False,False,False,302.009298,0.918285
3,ppo_selected_best,ppo_selected,4,61.324050,20.918285,0.358682,333.333347,-272.009298,20.743724,5.933585,False,False,False,302.009298,0.918285


In [5]:
def nav_return(frame: pd.DataFrame, col: str) -> float:
    if frame.empty or len(frame) < 2:
        return float('nan')
    return (float(frame[col].iloc[-1]) / float(frame[col].iloc[0]) - 1.0) * 100.0

def max_drawdown_pct(values: pd.Series) -> float:
    nav = values.astype(float)
    dd = nav / nav.cummax() - 1.0
    return float(-dd.min() * 100.0)

stage_defs = [
    ('pre_crash', '2020-01-02', '2020-02-21'),
    ('crash', '2020-02-24', '2020-03-23'),
    ('rebound', '2020-03-24', '2020-12-31'),
    ('full_year', '2020-01-02', '2020-12-31'),
]
reference_candidates = ['legacy_default', 'wf_pass_ytd18_15_15_02', 'floor_10_03_01', 'risk_cap_35_15_05']
rows = []
for candidate in reference_candidates:
    base = soxlbt_daily[(soxlbt_daily['candidate'].eq(candidate)) & (soxlbt_daily['window'].eq('wf_2020'))].copy()
    if base.empty:
        continue
    for stage, start, end in stage_defs:
        frame = base[(base['date'] >= start) & (base['date'] <= end)].copy()
        if len(frame) < 2:
            continue
        rows.append({
            'candidate': candidate,
            'stage': stage,
            'start': str(frame['date'].iloc[0].date()),
            'end': str(frame['date'].iloc[-1].date()),
            'rows': int(len(frame)),
            'strategy_return_pct': nav_return(frame, 'strategy_nav'),
            'buyhold_return_pct': nav_return(frame, 'buyhold_nav'),
            'strategy_max_dd_pct': max_drawdown_pct(frame['strategy_nav']),
            'avg_position': float(frame['position'].mean()),
            'max_position': float(frame['position'].max()),
            'buy_count': int(frame['action'].eq('BUY').sum()),
            'sell_count': int(frame['action'].isin(['SELL', 'TAKE_PROFIT']).sum()),
        })
soxlbt_stage = pd.DataFrame(rows)
soxlbt_stage.to_csv(CARD / 'soxlbacktest_wf2020_stage_metrics.csv', index=False)
soxlbt_stage

,candidate,stage,start,end,rows,strategy_return_pct,buyhold_return_pct,strategy_max_dd_pct,avg_position,max_position,buy_count,sell_count
0,legacy_default,pre_crash,2020-01-02,2020-02-21,35,0.000000,-2.914782,-0.000000,0.000000,0.0000,0,0
1,legacy_default,crash,2020-02-24,2020-03-23,21,-47.310881,-71.235752,54.275144,0.532648,0.9417,4,0
2,legacy_default,rebound,2020-03-24,2020-12-31,197,366.212353,397.631789,30.060943,0.959103,0.9813,0,0
3,legacy_default,full_year,2020-01-02,2020-12-31,253,220.777205,59.775308,54.275144,0.791023,0.9813,4,0
4,wf_pass_ytd18_15_15_02,pre_crash,2020-01-02,2020-02-21,35,0.000000,-2.914782,-0.000000,0.000000,0.0000,0,0
5,wf_pass_ytd18_15_15_02,crash,2020-02-24,2020-03-23,21,-9.512514,-71.235752,12.785207,0.135938,0.1591,1,0
6,wf_pass_ytd18_15_15_02,rebound,2020-03-24,2020-12-31,197,39.843240,397.631789,8.240690,0.205132,0.3337,1,0
7,wf_pass_ytd18_15_15_02,full_year,2020-01-02,2020-12-31,253,29.156603,59.775308,12.785207,0.171011,0.3337,2,0
8,floor_10_03_01,pre_crash,2020-01-02,2020-02-21,35,0.000000,-2.914782,-0.000000,0.000000,0.0000,0,0
9,floor_10_03_01,crash,2020-02-24,2020-03-23,21,-2.850950,-71.235752,3.731906,0.028790,0.0361,4,0


In [6]:
trades_2020 = soxlbt_trades[(soxlbt_trades['candidate'].eq('legacy_default')) & (soxlbt_trades['window'].eq('wf_2020'))].copy()
trades_2020['trigger_summary'] = trades_2020['trigger_conditions'].fillna('').str.slice(0, 180)
trades_2020[['date','type','price','position','score','buy_type','buy_step','gain','trigger_summary']].to_csv(CARD / 'soxlbacktest_legacy_wf2020_trades.csv', index=False)
trades_2020[['date','type','price','position','score','buy_type','buy_step','gain','trigger_summary']]

,date,type,price,position,score,buy_type,buy_step,gain,trigger_summary
0,2020-02-27,buy,12.742000,0.2999,-150.000000,cumulative,0.3,NaN,"[{'type': 'cumulative_drop', 'value': -0.41367..."
1,2020-03-09,buy,9.886000,0.5494,-150.000000,daily,0.3,NaN,"[{'type': 'daily_drop', 'value': -0.2533609203..."
2,2020-03-12,buy,6.802667,0.7562,-149.469066,daily,0.3,NaN,"[{'type': 'daily_drop', 'value': -0.3041461615..."
3,2020-03-13,buy,8.661333,0.9417,-98.661430,cumulative,0.3,NaN,"[{'type': 'cumulative_drop', 'value': -0.60144..."


In [7]:
def position_bucket(pos: float) -> str:
    if pos < 0.10:
        return '00_cash_lt10'
    if pos < 0.30:
        return '10_low_10_30'
    if pos < 0.50:
        return '30_mid_30_50'
    if pos < 0.70:
        return '50_high_50_70'
    return '70_full_ge70'

contract_rows = []
for label, frame in [('wf2020_contract', wf2020_contract), ('combined_contract', combined_contract)]:
    for fold, group in frame.groupby('fold'):
        risk_pos = group['risk_target_position'].astype(float)
        release = group[group['release_gate_state'].astype(bool)]
        downtrend = group[group['is_downtrend_state'].astype(bool)]
        rebound = group[group['is_rebound_cluster'].astype(bool)]
        contract_rows.append({
            'contract': label,
            'fold': fold,
            'rows': int(len(group)),
            'start': str(group['datetime'].min().date()),
            'end': str(group['datetime'].max().date()),
            'mean_risk_target_position': float(risk_pos.mean()),
            'median_risk_target_position': float(risk_pos.median()),
            'p90_risk_target_position': float(risk_pos.quantile(0.90)),
            'max_risk_target_position': float(risk_pos.max()),
            'rows_ge_50pct': int((risk_pos >= 0.50).sum()),
            'rows_ge_70pct': int((risk_pos >= 0.70).sum()),
            'share_ge_70pct': float((risk_pos >= 0.70).mean()),
            'release_rows': int(len(release)),
            'release_mean_position': float(release['risk_target_position'].mean()) if not release.empty else float('nan'),
            'release_ge_70pct_rows': int((release['risk_target_position'] >= 0.70).sum()) if not release.empty else 0,
            'rebound_rows': int(len(rebound)),
            'rebound_ge_70pct_rows': int((rebound['risk_target_position'] >= 0.70).sum()) if not rebound.empty else 0,
            'downtrend_rows': int(len(downtrend)),
            'downtrend_mean_position': float(downtrend['risk_target_position'].mean()) if not downtrend.empty else float('nan'),
        })
contract_audit = pd.DataFrame(contract_rows)
contract_audit.to_csv(CARD / 'target_contract_position_audit.csv', index=False)

bucket_rows = []
for label, frame in [('wf2020_contract', wf2020_contract), ('combined_contract', combined_contract)]:
    f = frame.copy()
    f['position_bucket'] = f['risk_target_position'].astype(float).map(position_bucket)
    bucket = f.groupby(['fold','position_bucket']).size().reset_index(name='rows')
    bucket['contract'] = label
    bucket_rows.append(bucket)
contract_buckets = pd.concat(bucket_rows, ignore_index=True)
contract_buckets.to_csv(CARD / 'target_contract_position_buckets.csv', index=False)
contract_audit

,contract,fold,rows,start,end,mean_risk_target_position,median_risk_target_position,p90_risk_target_position,max_risk_target_position,rows_ge_50pct,rows_ge_70pct,share_ge_70pct,release_rows,release_mean_position,release_ge_70pct_rows,rebound_rows,rebound_ge_70pct_rows,downtrend_rows,downtrend_mean_position
0,wf2020_contract,wf_2020_train,2470,2010-03-11,2019-12-31,0.312066,0.25,0.534500,0.915500,608,21,0.008502,42,0.571512,4,42,4,781,0.264986
1,combined_contract,wf_2020_train,2470,2010-03-11,2019-12-31,0.312066,0.25,0.534500,0.915500,608,21,0.008502,42,0.571512,4,42,4,781,0.264986
2,combined_contract,wf_2022_train,505,2020-01-02,2021-12-31,0.348817,0.25,0.742263,0.893943,114,66,0.130693,40,0.590498,17,66,22,234,0.252950
3,combined_contract,wf_2023_2024_train,756,2020-01-02,2022-12-30,0.277530,0.20,0.636353,0.893943,114,66,0.087302,40,0.590498,17,66,22,478,0.191079


In [8]:
selected_config = 'release10_rebound4_downtrend2_steps600'
slice_focus = bc_state_slice[bc_state_slice['config_name'].eq(selected_config)].copy()
slice_focus['fit_problem'] = slice_focus.apply(
    lambda row: 'weak_rebound_release_fit' if row['fold'] == 'wf_2020_train' and row['slice'] in {'release_gate_state','is_rebound_cluster'} and row['accuracy'] < 0.70 else 'none',
    axis=1,
)
slice_focus.to_csv(CARD / 'bc_slice_failure_focus.csv', index=False)
slice_focus[['fold','slice','samples','accuracy','label_unique_actions','predicted_unique_actions','label_dominant_action','label_dominant_share','predicted_dominant_action','predicted_dominant_share','slice_gate','fit_problem']]

,fold,slice,samples,accuracy,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share,slice_gate,fit_problem
30,wf_2020_train,all,2157,0.727399,5,3,4,0.313398,0,0.377376,True,none
31,wf_2020_train,drawdown_control_state,640,0.870313,4,3,2,0.596875,2,0.673438,True,none
32,wf_2020_train,is_downtrend_state,640,0.870313,4,3,2,0.596875,2,0.673438,True,none
33,wf_2020_train,is_rebound_cluster,36,0.611111,4,3,4,0.722222,4,0.611111,True,weak_rebound_release_fit
34,wf_2020_train,release_gate_state,36,0.611111,4,3,4,0.722222,4,0.611111,True,weak_rebound_release_fit
35,wf_2022_train,all,505,0.823762,5,4,4,0.304950,4,0.326733,True,none
36,wf_2022_train,drawdown_control_state,49,0.918367,3,2,4,0.857143,4,0.918367,True,none
37,wf_2022_train,is_downtrend_state,234,0.833333,4,4,1,0.431624,1,0.491453,True,none
38,wf_2022_train,is_rebound_cluster,66,0.803030,2,1,4,0.803030,4,1.000000,True,none
39,wf_2022_train,release_gate_state,40,1.000000,1,1,4,1.000000,4,1.000000,True,none


In [9]:
legacy_rebound = soxlbt_stage[(soxlbt_stage['candidate'].eq('legacy_default')) & (soxlbt_stage['stage'].eq('rebound'))].iloc[0]
legacy_full = soxlbt_stage[(soxlbt_stage['candidate'].eq('legacy_default')) & (soxlbt_stage['stage'].eq('full_year'))].iloc[0]
risk_floor_full = soxlbt_stage[(soxlbt_stage['candidate'].eq('wf_pass_ytd18_15_15_02')) & (soxlbt_stage['stage'].eq('full_year'))].iloc[0]
cap_full = soxlbt_stage[(soxlbt_stage['candidate'].eq('risk_cap_35_15_05')) & (soxlbt_stage['stage'].eq('full_year'))].iloc[0]
wf2020_selected = metric_gap[metric_gap['fold'].eq('wf_2020')].iloc[0]
wf2020_contract_row = contract_audit[(contract_audit['contract'].eq('wf2020_contract')) & (contract_audit['fold'].eq('wf_2020_train'))].iloc[0]
wf2020_release_slice = slice_focus[(slice_focus['fold'].eq('wf_2020_train')) & (slice_focus['slice'].eq('release_gate_state'))].iloc[0]

revision_rows = [
    {
        'proposal': 'do_not_run_full_retention_training_as_is',
        'reason': 'metric replay already has three-fold coverage but wf_2020 fails only on relative return; full training from the same target would repeat underparticipation.',
        'evidence': f"wf_2020 selected relative {wf2020_selected['selected_relative_return']:.2f}pp, MaxDD {wf2020_selected['selected_max_dd']:.2f}%",
        'allowed_next': False,
    },
    {
        'proposal': 'build_fold_local_bull_reentry_participation_target',
        'reason': 'legacy SOXLBacktest buy/sell captures the 2020 rebound but current cap10 target has sparse high-participation labels.',
        'evidence': f"legacy rebound avg_pos {legacy_rebound['avg_position']:.3f}; wf2020 target share_ge_70pct {wf2020_contract_row['share_ge_70pct']:.3f}; release slice accuracy {wf2020_release_slice['accuracy']:.3f}",
        'allowed_next': True,
    },
    {
        'proposal': 'separate_reference_roles',
        'reason': 'legacy_default is a buy/sell timing teacher but not a risk-retention model; low-risk SOXLBacktest probes are drawdown floor references only.',
        'evidence': f"legacy full return {legacy_full['strategy_return_pct']:.2f}% / MaxDD {legacy_full['strategy_max_dd_pct']:.2f}%; risk floor return {risk_floor_full['strategy_return_pct']:.2f}% / MaxDD {risk_floor_full['strategy_max_dd_pct']:.2f}%; capped return {cap_full['strategy_return_pct']:.2f}% / MaxDD {cap_full['strategy_max_dd_pct']:.2f}%",
        'allowed_next': True,
    },
    {
        'proposal': 'keep_existing_drift_stop_rules',
        'reason': 'PPO improves some fold returns but prior screens show drawdown drift; stop rules must stay before actor retention.',
        'evidence': 'ppo_vs_bc_max_dd_delta_pp <= 2.0 and ppo_vs_bc_relative_delta_pp >= -5.0 remain required controls.',
        'allowed_next': True,
    },
]
training_revision = pd.DataFrame(revision_rows)
training_revision.to_csv(CARD / 'training_objective_revision.csv', index=False)
training_revision

,proposal,reason,evidence,allowed_next
0,do_not_run_full_retention_training_as_is,metric replay already has three-fold coverage ...,"wf_2020 selected relative -292.75pp, MaxDD 14.98%",False
1,build_fold_local_bull_reentry_participation_ta...,legacy SOXLBacktest buy/sell captures the 2020...,legacy rebound avg_pos 0.959; wf2020 target sh...,True
2,separate_reference_roles,legacy_default is a buy/sell timing teacher bu...,legacy full return 220.78% / MaxDD 54.28%; ris...,True
3,keep_existing_drift_stop_rules,PPO improves some fold returns but prior scree...,ppo_vs_bc_max_dd_delta_pp <= 2.0 and ppo_vs_bc...,True


In [10]:
failed_fold = 'wf_2020'
all_metric_folds = set(metric_gap['fold'])
required_folds = {'wf_2020', 'wf_2022', 'wf_2023_2024'}
failed_folds = metric_gap.loc[~metric_gap['selected_candidate_gate'].astype(bool), 'fold'].tolist()

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'wf2020_failed_fold_requires_bull_reentry_target_revision_no_promotion',
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'cap10_three_fold_checkpoint_metric_replay_v2',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
    'changed_layer': ['model_selection', 'training_objective', 'target_design'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'metric_replay_folds_present': sorted(all_metric_folds),
    'missing_required_folds': sorted(required_folds - all_metric_folds),
    'failed_folds': failed_folds,
    'primary_failure': 'wf_2020_relative_underparticipation_after_metric_replay',
    'wf_2020_selected': {
        'candidate': str(wf2020_selected['selected_candidate']),
        'relative_return_pp': float(wf2020_selected['selected_relative_return']),
        'max_dd_pct': float(wf2020_selected['selected_max_dd']),
        'relative_gap_to_target_pp': float(wf2020_selected['relative_gap_to_target_pp']),
        'maxdd_gap_to_target_pp': float(wf2020_selected['maxdd_gap_to_target_pp']),
    },
    'soxlbacktest_reference_readout': {
        'legacy_default_full_return_pct': float(legacy_full['strategy_return_pct']),
        'legacy_default_full_max_dd_pct': float(legacy_full['strategy_max_dd_pct']),
        'legacy_default_rebound_return_pct': float(legacy_rebound['strategy_return_pct']),
        'legacy_default_rebound_avg_position': float(legacy_rebound['avg_position']),
        'risk_floor_full_return_pct': float(risk_floor_full['strategy_return_pct']),
        'risk_floor_full_max_dd_pct': float(risk_floor_full['strategy_max_dd_pct']),
        'risk_cap_35_15_05_full_return_pct': float(cap_full['strategy_return_pct']),
        'risk_cap_35_15_05_full_max_dd_pct': float(cap_full['strategy_max_dd_pct']),
    },
    'target_contract_readout': {
        'wf2020_target_rows': int(wf2020_contract_row['rows']),
        'wf2020_share_ge_70pct': float(wf2020_contract_row['share_ge_70pct']),
        'wf2020_rows_ge_70pct': int(wf2020_contract_row['rows_ge_70pct']),
        'wf2020_release_rows': int(wf2020_contract_row['release_rows']),
        'wf2020_release_mean_position': float(wf2020_contract_row['release_mean_position']),
        'wf2020_release_ge_70pct_rows': int(wf2020_contract_row['release_ge_70pct_rows']),
        'wf2020_release_slice_accuracy': float(wf2020_release_slice['accuracy']),
    },
    'training_allowed_from_current_cap10_as_is': False,
    'next_training_allowed_after_target_revision': True,
    'next_mechanism': 'build_fold_local_bull_reentry_participation_target_then_rerun_three_fold_bc_checkpoint_metric_replay',
    'required_next_run_controls': [
        'derive bull/reentry labels fold-locally from train windows only',
        'keep SOXLBacktest legacy_default as buy/sell timing reference, not risk-retention model',
        'keep low-risk SOXLBacktest variants as drawdown floor references only',
        'require wf_2020 relative >= +30pp and MaxDD < 20% before full retention training',
        'require wf_2022 and wf_2023_2024 gates remain passed',
        'apply PPO-vs-BC MaxDD drift stop <= 2pp and relative delta >= -5pp',
        'export per-model dashboard feature snapshots and BUY/SELL feature summaries only after a retained actor exists',
    ],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'sidecars': [
        'wf_fold_metric_gap.csv',
        'wf2020_checkpoint_candidate_gap.csv',
        'soxlbacktest_wf2020_stage_metrics.csv',
        'soxlbacktest_legacy_wf2020_trades.csv',
        'target_contract_position_audit.csv',
        'target_contract_position_buckets.csv',
        'bc_slice_failure_focus.csv',
        'training_objective_revision.csv',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}

summary = {
    **selection,
    'conclusion': [
        'Do not start full retained cap10 training from the current target as-is.',
        'After qlib metric replay, wf_2020 is the only failed fold; its MaxDD is already under 20%, but relative return is deeply negative because the candidate under-participates in the post-crash bull rebound.',
        'SOXLBacktest legacy_default is useful as a buy/sell timing reference because it captures the 2020 rebound, but it is not promotable because full-year drawdown is too high.',
        'Low-risk SOXLBacktest variants show how to pass drawdown by reducing exposure, but their rebound participation is too low to satisfy return alignment.',
        'The next training objective should add a fold-local bull/reentry participation target before rerunning BC/checkpoint metric replay; keep existing drift stop rules and block actor retention until all folds pass.',
    ],
}

(CARD / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True) + chr(10), encoding='utf-8')
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + chr(10), encoding='utf-8')

conclusion_lines = [
    '# Cap10 WF2020 Failed Fold Objective Audit',
    '',
    f"- Decision: `{selection['decision']}`.",
    '- Promotion: blocked; no actor is promoted or retained by this diagnostic.',
    f"- Failed fold: `wf_2020`; selected `{selection['wf_2020_selected']['candidate']}` has relative `{selection['wf_2020_selected']['relative_return_pp']:.2f}pp` and MaxDD `{selection['wf_2020_selected']['max_dd_pct']:.2f}%`.",
    f"- SOXLBacktest `legacy_default` captures the 2020 rebound with rebound return `{selection['soxlbacktest_reference_readout']['legacy_default_rebound_return_pct']:.2f}%` and avg position `{selection['soxlbacktest_reference_readout']['legacy_default_rebound_avg_position']:.3f}`, but full-year MaxDD is `{selection['soxlbacktest_reference_readout']['legacy_default_full_max_dd_pct']:.2f}%`.",
    f"- Current wf2020 target contract has only `{selection['target_contract_readout']['wf2020_rows_ge_70pct']}` rows at `>=70%` target position out of `{selection['target_contract_readout']['wf2020_target_rows']}` rows; release slice accuracy is `{selection['target_contract_readout']['wf2020_release_slice_accuracy']:.3f}`.",
    '- Next action: build a fold-local bull/reentry participation target, then rerun three-fold BC/checkpoint metric replay before any full retention training.',
    '- Dashboard note: do not expose this as runnable in v2; dashboard feature snapshots and BUY/SELL summaries should be exported only after a retained actor exists.',
]
(CARD / 'conclusion.md').write_text(chr(10).join(conclusion_lines) + chr(10), encoding='utf-8')

summary

{'experiment': 'cap10_wf2020_failed_fold_objective_audit_v1',
 'created_at': '2026-06-13T20:53:31.586999+00:00',
 'decision': 'wf2020_failed_fold_requires_bull_reentry_target_revision_no_promotion',
 'role': 'diagnose',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'cap10_three_fold_checkpoint_metric_replay_v2',
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
 'changed_layer': ['model_selection', 'training_objective', 'target_design'],
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 20.0,
 'metric_replay_folds_present': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'missing_required_folds': [],
 'failed_folds': ['wf_2020'],
 'primary_failure': 'wf_2020_relative_underparticipation_after_metric_replay',
 'wf_2020_selected': {'candidate': 'bc_initialization',
  'relative_return_pp': -292.75302146825334,
  'max_dd_pct': 14.984700509769244,
  'relative_gap_to_target_pp': 322.75302146825334,
  'maxdd_gap_to_target_pp': 0.0},

## Conclusion

Decision: `wf2020_failed_fold_requires_bull_reentry_target_revision_no_promotion`.

The current cap10 branch should not enter full retained training as-is. After qlib metric replay, `wf_2020` is the only failed fold; the selected BC initialization already keeps MaxDD under 20%, but relative return remains deeply negative because the target under-participates in the post-crash bull rebound.

SOXLBacktest `legacy_default` is useful as a buy/sell timing reference, not as a promotable risk model: it captures the 2020 rebound but carries excessive full-year drawdown. Low-risk SOXLBacktest variants prove the drawdown floor, but they destroy rebound participation. The next training objective should add a fold-local bull/reentry participation target, rerun the three-fold BC/checkpoint metric replay, and keep actor retention blocked until all walk-forward, YTD return-alignment, and dashboard replay gates pass.